# RAG Evaluation Framework
This notebook implements a RAG framework using LangChain, Chroma, and Ollama, and evaluates it against the `allganize/RAG-Evaluation-Dataset-KO` dataset.

In [29]:
import json
import os
import pandas as pd

json_dir = r"C:/Users/USER/Downloads/BioASQ-training4b"

json_path = os.path.join(json_dir, "BioASQ-trainingDataset4b.json")

# 1) JSON 로드
with open(json_path, "r", encoding="utf-8") as f:
    data = json.load(f)

# 2) 질문 리스트 꺼내기
# (BioASQ training은 보통 questions 키)
questions = data.get("questions") or data.get("question")

print("질문 수:", len(questions))

# 3) DataFrame 변환
df = pd.DataFrame(questions)

# 4) triples / concepts 제거
df = df.drop(columns=["triples", "concepts"], errors="ignore")

print(df.columns)
df


질문 수: 1307
Index(['body', 'documents', 'ideal_answer', 'type', 'id', 'snippets',
       'exact_answer'],
      dtype='object')


,body,documents,ideal_answer,type,id,snippets,exact_answer
0,What symptoms characterize the Muenke syndrome?,"[http://www.ncbi.nlm.nih.gov/pubmed/23378035, ...",[Muenke syndrome is characterized by considera...,summary,52bf1d3c03868f1b0600000d,"[{'offsetInBeginSection': 164, 'offsetInEndSec...",NaN
1,What is the inheritance pattern of Li–Fraumeni...,"[http://www.ncbi.nlm.nih.gov/pubmed/7981072, h...",[Li-Fraumeni syndrome shows autosomal dominant...,factoid,52bf208003868f1b06000019,"[{'offsetInBeginSection': 524, 'offsetInEndSec...",[[Autosomal dominant]]
2,What are the indications for alteplase?,"[http://www.ncbi.nlm.nih.gov/pubmed/23727456, ...",[Intravenous alteplase (recombinant tissue pla...,list,52bf209303868f1b0600001a,"[{'offsetInBeginSection': 0, 'offsetInEndSecti...","[ischemic stroke, central venous catheter (CVC..."
3,What is the role of Inn1 in cytokinesis?,"[http://www.ncbi.nlm.nih.gov/pubmed/22956544, ...",[Inn1 associates with the contractile actomyos...,summary,530cf22aa177c6630c000001,"[{'offsetInBeginSection': 0, 'offsetInEndSecti...",NaN
4,What is the main role of Ctf4 in dna replication?,"[http://www.ncbi.nlm.nih.gov/pubmed/24255107, ...",[Ctf4 coordinates the progression of helicase ...,factoid,530cf22aa177c6630c000004,"[{'offsetInBeginSection': 95, 'offsetInEndSect...",[[Coordination of the progression of helicase ...
...,...,...,...,...,...,...,...
1302,Which is the receptor for substrates of Chaper...,"[http://www.ncbi.nlm.nih.gov/pubmed/23404999, ...",[Chaperone-mediated autophagy (CMA) is a lysos...,factoid,51bdd9c2047fa84d1d000002,"[{'offsetInBeginSection': 149, 'offsetInEndSec...","[LAMP2A, Lysosome-associated membrane protein ..."
1303,Which are the Atg8 homologs in human?,"[http://www.ncbi.nlm.nih.gov/pubmed/23043107, ...",[Autophagy (Autophagy-related protein 8 or Atg...,list,51bdf045047fa84d1d000003,"[{'offsetInBeginSection': 482, 'offsetInEndSec...","[[MAP1LC3A, microtubule-associated protein-1 l..."
1304,Describe the known functions for the prothymos...,"[http://www.ncbi.nlm.nih.gov/pubmed/23201434, ...",[Prothymosin alpha (ProTα) (encoded in human b...,summary,51be03c4047fa84d1d000004,"[{'offsetInBeginSection': 0, 'offsetInEndSecti...",NaN
1305,Has field-programmable gate array (FPGA) techn...,"[http://www.ncbi.nlm.nih.gov/pubmed/22151470, ...",[Yes. Field-Programmable Gate Arrays (FPGAs) a...,yesno,51be1750047fa84d1d000005,"[{'offsetInBeginSection': 868, 'offsetInEndSec...",yes


In [10]:
df['type'].value_counts()

type
yesno      368
factoid    327
list       327
summary    285
Name: count, dtype: int64

In [11]:
# summary 제거
df = df[df["type"] != "summary"].reset_index(drop=True)

print("summary 제거 후 질문 수:", len(df))
print(df["type"].value_counts())


summary 제거 후 질문 수: 1022
type
yesno      368
factoid    327
list       327
Name: count, dtype: int64


In [12]:
prefix = "http://www.ncbi.nlm.nih.gov/pubmed/"

df["documents"] = df["documents"].apply(
    lambda lst: [x.replace(prefix, "") for x in lst] if isinstance(lst, list) else lst
)


In [13]:
prefix = "http://www.ncbi.nlm.nih.gov/pubmed/"

def strip_snippet_docs(snips):
    if not isinstance(snips, list):
        return snips
    for s in snips:
        if isinstance(s, dict) and "document" in s and isinstance(s["document"], str):
            s["document"] = s["document"].replace(prefix, "")
    return snips

df["snippets"] = df["snippets"].apply(strip_snippet_docs)


In [16]:
from pathlib import Path
yesno_df = df[df["type"] == "factoid"]
out_dir = Path(r"C:/Users/USER\Desktop/RAG_Korean/data")
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "bioasq_factoid.csv"
yesno_df.to_csv(out_path, index=False, encoding="utf-8-sig")

In [32]:
data = pd.read_csv("C:/Users/USER\Desktop/RAG_Korean/data/bioasq_list.csv")

In [33]:
# 1) 데이터 전체에 NaN이 하나라도 있는지 (True/False)
print(data.isna().any().any())

# 2) 컬럼별 NaN 개수
print(data.isna().sum().sort_values(ascending=False))

# 3) NaN이 있는 행(row) 개수
print(data.isna().any(axis=1).sum())

# 4) NaN이 있는 행만 보고 싶을 때
print(data[data.isna().any(axis=1)])


False
body            0
documents       0
ideal_answer    0
type            0
id              0
snippets        0
exact_answer    0
dtype: int64
0
Empty DataFrame
Columns: [body, documents, ideal_answer, type, id, snippets, exact_answer]
Index: []


In [88]:
df['exact_answer'][0]

[['Autosomal dominant']]

In [34]:
len(df)

1022

In [19]:
df['type'].value_counts()

type
yesno      368
factoid    327
list       327
Name: count, dtype: int64

In [24]:
import pandas as pd
import numpy as np
import ast

# documents가 문자열로 저장돼 있을 수도 있어서 안전하게 list로 변환
def to_list(x):
    if isinstance(x, list):
        return x
    if pd.isna(x):
        return []
    if isinstance(x, str):
        x = x.strip()
        if x == "" or x.lower() == "nan":
            return []
        try:
            v = ast.literal_eval(x)         # "[1,2,3]" -> [1,2,3]
            return v if isinstance(v, list) else [v]
        except Exception:
            # 혹시 literal_eval이 실패하면 콤마 split로 fallback
            return [t.strip() for t in x.split(",") if t.strip()]
    return [x]

docs_list = df["documents"].apply(to_list)
doc_cnt = docs_list.apply(len)   # 각 row에 문서 몇 개인지

# 1) 사분위수(Q1/Q2/Q3) 값 요약
q1, q2, q3 = doc_cnt.quantile([0.25, 0.5, 0.75]).tolist()
summary = {
    "count": int(doc_cnt.count()),
    "mean": float(doc_cnt.mean()),
    "min": int(doc_cnt.min()),
    "Q1(25%)": float(q1),
    "Q2(50%, median)": float(q2),
    "Q3(75%)": float(q3),
    "max": int(doc_cnt.max()),
    "IQR(Q3-Q1)": float(q3 - q1),
}
print(pd.Series(summary))



count              1022.000000
mean                 13.634051
min                   1.000000
Q1(25%)               6.000000
Q2(50%, median)      11.000000
Q3(75%)              18.000000
max                 157.000000
IQR(Q3-Q1)           12.000000
dtype: float64


In [30]:
df['documents'][0]

['http://www.ncbi.nlm.nih.gov/pubmed/23378035',
 'http://www.ncbi.nlm.nih.gov/pubmed/22446440',
 'http://www.ncbi.nlm.nih.gov/pubmed/22016144',
 'http://www.ncbi.nlm.nih.gov/pubmed/21844411',
 'http://www.ncbi.nlm.nih.gov/pubmed/21403567',
 'http://www.ncbi.nlm.nih.gov/pubmed/20592905',
 'http://www.ncbi.nlm.nih.gov/pubmed/20301588',
 'http://www.ncbi.nlm.nih.gov/pubmed/18000976',
 'http://www.ncbi.nlm.nih.gov/pubmed/17414289',
 'http://www.ncbi.nlm.nih.gov/pubmed/14963686',
 'http://www.ncbi.nlm.nih.gov/pubmed/23044018',
 'http://www.ncbi.nlm.nih.gov/pubmed/20301628',
 'http://www.ncbi.nlm.nih.gov/pubmed/22622662']

In [49]:
df['snippets'][0]

[{'offsetInBeginSection': 524,
  'offsetInEndSection': 745,
  'text': 'It therefore appears that the LFS phenotype has been conferred by an aberrant gene, showing a dominant pattern of inheritance, which may be acting to compromise normal p53 function rather than by a mutation in p53 itself.',
  'beginSection': 'abstract',
  'document': '7981072',
  'endSection': 'abstract'},
 {'offsetInBeginSection': 725,
  'offsetInEndSection': 1213,
  'text': 'In addition, there seem to be predispositions to a wider range of different, but well-defined neoplasms: e.g., adenocarcinomatosis of the colon and the endometrium, or the Li-Fraumeni/SBLA syndrome. The latter shows a spectrum of sarcoma, brain tumours, breast cancer, leukaemias, lung and adenocortical cancer. The genes leading to these types of dominantly inherited predispositions appear to be the tentatively so-called tumour suppressor genes, for which the Rb gene serves as a model',
  'beginSection': 'abstract',
  'document': '2190528',
  '

In [7]:
import re

def extract_pmid_from_url(url: str):
    if url is None:
        return None
    m = re.search(r"/pubmed/(\d+)", str(url))
    return m.group(1) if m else None

# documents 컬럼(각 행: URL list)을 전부 평탄화(flatten)해서 PMID 뽑기
all_pmids = set()

for doc_list in df["documents"]:
    if not isinstance(doc_list, list):
        continue
    for url in doc_list:
        pmid = extract_pmid_from_url(url)
        if pmid:
            all_pmids.add(pmid)

pmid_list = sorted(list(all_pmids))
print("고유 PMID 수:", len(pmid_list))
print("샘플:", pmid_list[:10])


고유 PMID 수: 13072
샘플: ['10024311', '10024886', '10026129', '1002690', '10027003', '10027580', '10029885', '10030434', '10030577', '10063835']


In [14]:
%pip install biopython

   ---------------------------------------- 0.0/2.7 MB ? eta -:--:--
   ---------------------------------------- 2.7/2.7 MB 52.6 MB/s  0:00:00
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# pubmed 크롤링
from Bio import Entrez
import pandas as pd
import time

Entrez.email = "jibig12@kmu.kr"
Entrez.tool = "MyPubMedScript"

def fetch_pubmed_title_abstract(pmid_list):
    titles, abstracts, pmids_done = [], [], []

    BATCH_SIZE = 50
    for i in range(0, len(pmid_list), BATCH_SIZE):
        batch_pmids = pmid_list[i:i+BATCH_SIZE]

        handle = Entrez.efetch(
            db="pubmed",
            id=",".join(batch_pmids),
            retmode="xml"
        )
        records = Entrez.read(handle)
        handle.close()

        for rec in records.get("PubmedArticle", []):
            pmid = str(rec["MedlineCitation"]["PMID"])
            pmids_done.append(pmid)

            # Title
            title = rec["MedlineCitation"]["Article"].get("ArticleTitle", None)

            # Abstract
            abstract = None
            try:
                abs_obj = rec["MedlineCitation"]["Article"]["Abstract"]["AbstractText"]
                # abs_obj는 list일 수도, str 비슷한 객체일 수도 있음
                if isinstance(abs_obj, list):
                    abstract = " ".join([str(x) for x in abs_obj])
                else:
                    abstract = str(abs_obj)
            except Exception:
                abstract = None

            titles.append(str(title) if title is not None else None)
            abstracts.append(abstract)

        time.sleep(0.34)  # NCBI rate limit 대비(너무 빠르면 막힘)

    return pd.DataFrame({"pmid": pmids_done, "title": titles, "abstract": abstracts})


In [93]:
df_pubmed = fetch_pubmed_title_abstract(pmid_list)

# 결측치 제거(abstract 없으면 RAG에 쓸 가치 낮음)
df_pubmed = df_pubmed.dropna(subset=["abstract"]).reset_index(drop=True)

# 문서 텍스트 만들기
df_pubmed["text"] = df_pubmed.apply(
    lambda r: f"Title: {r['title']}\nAbstract: {r['abstract']}",
    axis=1
)

print(df_pubmed.head(3))

# 저장
df_pubmed.to_csv("pubmed_title_abstract.csv", index=False, encoding="utf-8")
print("저장 완료: pubmed_title_abstract.csv")


       pmid                                              title  \
0  10024311  Phospholamban is present in endothelial cells ...   
1  10024886  Virus infection leads to localized hyperacetyl...   
2  10026129  X inactive-specific transcript (Xist) expressi...   

                                            abstract  \
0  Vascular endothelial cells regulate vascular s...   
1  Transcriptional activation of the human interf...   
2  Expression of the X inactive-specific transcri...   

                                                text  
0  Title: Phospholamban is present in endothelial...  
1  Title: Virus infection leads to localized hype...  
2  Title: X inactive-specific transcript (Xist) e...  
저장 완료: pubmed_title_abstract.csv


In [ ]:
df_pubmed

,pmid,title,abstract,text
0,10024311,Phospholamban is present in endothelial cells ...,Vascular endothelial cells regulate vascular s...,Title: Phospholamban is present in endothelial...
1,10024886,Virus infection leads to localized hyperacetyl...,Transcriptional activation of the human interf...,Title: Virus infection leads to localized hype...
2,10026129,X inactive-specific transcript (Xist) expressi...,Expression of the X inactive-specific transcri...,Title: X inactive-specific transcript (Xist) e...
3,1002690,mRNA guanylyltransferase and mRNA (guanine-7-)...,Characterization of the donor and acceptor spe...,Title: mRNA guanylyltransferase and mRNA (guan...
4,10027003,Mechanisms of development of multiple endocrin...,The ret proto-oncogene encodes a receptor tyro...,Title: Mechanisms of development of multiple e...
...,...,...,...,...
13002,9973290,Mapping of a familial moyamoya disease gene to...,Moyamoya disease is characterized by bilateral...,Title: Mapping of a familial moyamoya disease ...
13003,9973956,Cytogenetic changes in two cases of subependym...,Cytogenetic analysis of subependymal giant-cel...,Title: Cytogenetic changes in two cases of sub...
13004,9989412,The transcriptional cofactor complex CRSP is r...,Activation of gene transcription in metazoans ...,Title: The transcriptional cofactor complex CR...
13005,9990116,CpG islands in chromatin organization and gene...,CpG islands are stretches of DNA sequence that...,Title: CpG islands in chromatin organization a...


: 

In [ ]:
df_pubmed['abstract'][0]

'To assess quality of storage of vaccines in the community. Questionnaire survey of general practices and child health clinics, and monitoring of storage temperatures of selected refrigerators. Central Manchester and Bradford health districts. 45 general practices and five child health clinics, of which 40 (80%) responded. Eight practices were selected for refrigeration monitoring. Adherence to Department of Health guidelines for vaccine storage, temperature range to which vaccines were exposed over two weeks. Of the 40 respondents, only 16 were aware of the appropriate storage conditions for the vaccines; eight had minimum and maximum thermometers but only one of these was monitored daily. In six of the eight practices selected for monitoring of refrigeration temperatures the vaccines were exposed to either subzero temperatures (three fridges) or temperatures up to 16 degrees C (three). Two of these were specialised drug storage refrigerators with an incorporated thermostat and extern

In [35]:
import os
import pandas as pd
from datasets import load_dataset
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma
from langchain_community.chat_models import ChatOllama
from langchain_core.prompts import load_prompt # 👈 core 모듈에서 가져오세요
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from tqdm import tqdm
import glob
from huggingface_hub import login

from sentence_transformers import SentenceTransformer

## 1. 데이터 불러오기

In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import PyMuPDFLoader
from langchain_community.vectorstores import FAISS
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough
from langchain_core.prompts import PromptTemplate
from langchain_core.documents import Document
from langchain.embeddings.base import Embeddings

c:\Users\USER\anaconda3\envs\spacy38\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [36]:
df = pd.read_csv(r"C:/Users/USER/Desktop/RAG_Korean/pubmed_title_abstract.csv")
df

,pmid,title,abstract,text
0,10024311,Phospholamban is present in endothelial cells ...,Vascular endothelial cells regulate vascular s...,Title: Phospholamban is present in endothelial...
1,10024886,Virus infection leads to localized hyperacetyl...,Transcriptional activation of the human interf...,Title: Virus infection leads to localized hype...
2,10026129,X inactive-specific transcript (Xist) expressi...,Expression of the X inactive-specific transcri...,Title: X inactive-specific transcript (Xist) e...
3,1002690,mRNA guanylyltransferase and mRNA (guanine-7-)...,Characterization of the donor and acceptor spe...,Title: mRNA guanylyltransferase and mRNA (guan...
4,10027003,Mechanisms of development of multiple endocrin...,The ret proto-oncogene encodes a receptor tyro...,Title: Mechanisms of development of multiple e...
...,...,...,...,...
13002,9973290,Mapping of a familial moyamoya disease gene to...,Moyamoya disease is characterized by bilateral...,Title: Mapping of a familial moyamoya disease ...
13003,9973956,Cytogenetic changes in two cases of subependym...,Cytogenetic analysis of subependymal giant-cel...,Title: Cytogenetic changes in two cases of sub...
13004,9989412,The transcriptional cofactor complex CRSP is r...,Activation of gene transcription in metazoans ...,Title: The transcriptional cofactor complex CR...
13005,9990116,CpG islands in chromatin organization and gene...,CpG islands are stretches of DNA sequence that...,Title: CpG islands in chromatin organization a...


In [37]:
import pandas as pd

df = pd.read_csv(r"C:/Users/USER/Desktop/RAG_Korean/pubmed_title_abstract.csv")

text = df["text"].fillna("").astype(str)   # ✅ text 컬럼 직접 사용

char_len = text.str.len()
word_len = text.str.split().str.len()

def quartile_summary(s: pd.Series) -> pd.Series:
    q1, q2, q3 = s.quantile([0.25, 0.5, 0.75]).tolist()
    return pd.Series({
        "count": int(s.count()),
        "mean": float(s.mean()),
        "min": float(s.min()),
        "Q1(25%)": float(q1),
        "Q2(50%, median)": float(q2),
        "Q3(75%)": float(q3),
        "max": float(s.max()),
        "IQR(Q3-Q1)": float(q3 - q1),
    })

summary = pd.DataFrame({
    "chars": quartile_summary(char_len),
    "words": quartile_summary(word_len),
}).T

print(summary)


         count         mean    min  Q1(25%)  Q2(50%, median)  Q3(75%)     max  \
chars  13007.0  1547.616360  117.0   1219.0           1544.0   1838.0  9686.0   
words  13007.0   221.488122   13.0    173.0            220.0    264.0  1346.0   

       IQR(Q3-Q1)  
chars       619.0  
words        91.0  


In [2]:
import pandas as pd
df = pd.read_csv('C:/Users/USER/Desktop/RAG_Korean/pubmed_title_abstract.csv')
documents = df.to_dict(orient="records")

In [3]:
documents

[{'pmid': 10024311,
  'title': 'Phospholamban is present in endothelial cells and modulates endothelium-dependent relaxation. Evidence from phospholamban gene-ablated mice.',
  'abstract': 'Vascular endothelial cells regulate vascular smooth muscle tone through Ca2+-dependent production and release of vasoactive molecules. Phospholamban (PLB) is a 24- to 27-kDa phosphoprotein that modulates activity of the sarco(endo)plasmic reticulum Ca2+ ATPase (SERCA). Expression of PLB is reportedly limited to cardiac, slow-twitch skeletal and smooth muscle in which PLB is an important regulator of [Ca2+]i and contractility in these muscles. In the present study, we report the existence of PLB in the vascular endothelium, a nonmuscle tissue, and provide functional data on PLB regulation of vascular contractility through its actions in the endothelium. Endothelium-dependent relaxation to acetylcholine was attenuated in aorta of PLB-deficient (PLB-KO) mice compared with wild-type (WT) controls. This 

In [4]:
document_list = [
    Document(
        page_content=d["text"],
        metadata={
            "pmid": str(d.get("pmid", "")),
            "title":  d.get("title", ""),
            "id":     d.get("id", i),        # 혹시 id 없으면 인덱스라도
        }
    )
    for i, d in enumerate(documents)
]

In [5]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)
splits = text_splitter.split_documents(document_list)
print(f"Created {len(splits)} chunks.")

Created 63009 chunks.


In [6]:
splits[10]

Document(metadata={'pmid': '10026129', 'title': 'X inactive-specific transcript (Xist) expression and X chromosome inactivation in the preattachment bovine embryo.', 'id': 2}, page_content='Abstract: Expression of the X inactive-specific transcript (Xist) is thought to be essential for the initiation of X chromosome inactivation and dosage compensation during female embryo development. In the present study, we analyzed the patterns of Xist transcription and the onset of X chromosome inactivation in bovine preattachment embryos. Reverse transcription-polymerase chain reaction (RT-PCR) revealed the presence of Xist transcripts in all adult female somatic tissues evaluated. In')

## 2. Create Vector Database (Chroma)

In [25]:
from chromadb import PersistentClient

PERSIST_DIR = "C:/Users/USER/Desktop/RAG_Korean/500-100/embeddinggemma/chroma_db"

# Chroma 클라이언트 연결
client = PersistentClient(path=PERSIST_DIR)

# 전체 컬렉션 목록 조회
collections = client.list_collections()

print("📚 존재하는 Collection 목록:")
for c in collections:
    print(" -", c.name)

📚 존재하는 Collection 목록:
 - gemma_M
 - gemma_X
 - gemma_C


## 3.RAG

In [13]:
%pip install -U accelerate

Note: you may need to restart the kernel to use updated packages.


In [26]:
# ============================================================
# BioASQ RAG Evaluator (single script, with main)
# total rag
# - uses existing Chroma (NO re-index / NO re-embedding)
# - embedding_function 생성 방식을 통일
# - Prompt: ✅ [SYSTEM]/[User] 분리 + SystemMessage/HumanMessage로 invoke (논문 Table 3)
# - ✅ 결과 CSV에 gold/pred exact & ideal을 함께 저장(비교용)
# ============================================================

import os, time, ast, json, re, unicodedata, random
import numpy as np
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import ndcg_score

import torch
from torch import nn
from sentence_transformers import SentenceTransformer, models

from langchain_chroma import Chroma
from langchain_community.chat_models import ChatOllama

# (LangChain Embeddings import) - 버전 차이 대응
try:
    from langchain.embeddings.base import Embeddings
except Exception:
    from langchain_core.embeddings import Embeddings

# ✅ messages role 분리용
try:
    from langchain_core.messages import SystemMessage, HumanMessage
except Exception:
    from langchain.schema import SystemMessage, HumanMessage

# ✅ YAML 읽기
try:
    import yaml
except Exception as e:
    raise ImportError("pyyaml가 필요합니다. `pip install pyyaml` 후 다시 실행하세요.") from e


# =========================
# 0) 설정 (너 환경에 맞게)
# =========================
DATASET_LIST_CSV = "C:/Users/USER/Desktop/RAG_Korean/data/bioasq_factoid.csv"
CHROMA_DIR = "C:/Users/USER/Desktop/RAG_Korean/500-100/embeddinggemma/chroma_db"
PROMPT_PATH = "C:/Users/USER/Desktop/RAG_Korean/prompt/rag_prompt2.yaml"

RESULTS_DIR = "C:/Users/USER/Desktop/RAG_Korean/500-100/result"
SUMMARY_DIR = "C:/Users/USER/Desktop/RAG_Korean/500-100/result_summary"
os.makedirs(RESULTS_DIR, exist_ok=True)
os.makedirs(SUMMARY_DIR, exist_ok=True)

collection_list = ["gemma_C", "gemma_X","gemma_M" ]
k_list = [5,10,20]

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

LLM_MODEL = "gemma3:4b"
MAX_DOC_CHARS = 1000
MAX_TOTAL_CHARS = 40000

# LLM이 sources를 안 주면, 검색 결과를 evidence로 대체할지
EVIDENCE_FALLBACK_TO_RETRIEVAL = True

# ✅ 디버그를 콘솔 도배하지 않고 CSV로 저장
DEBUG_SAVE_FIRST_N = 50   # 처음 N개 질문만 디버그 저장(0이면 끔)

# =====================================================
# (선택) Seed 고정
# =====================================================
SEED = 42
os.environ["PYTHONHASHSEED"] = str(SEED)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)


# =========================
# 1) 통합 풀링 모듈
# =========================
class UnifiedConcatPooling(nn.Module):
    def __init__(self, word_embedding_dimension: int,
                 use_first=False, use_last=False, use_mean=False, use_max=False):
        super().__init__()
        self.word_embedding_dimension = word_embedding_dimension
        self.use_first = use_first
        self.use_last  = use_last
        self.use_mean  = use_mean
        self.use_max   = use_max

    def forward(self, features):
        token_embeddings = features["token_embeddings"]  # [B,T,H]
        attention_mask  = features["attention_mask"]     # [B,T]

        outs = []

        if self.use_first:
            outs.append(token_embeddings[:, 0])

        if self.use_last:
            last_idx = attention_mask.long().sum(dim=1) - 1
            last_idx = torch.clamp(last_idx, min=0)
            bidx = torch.arange(token_embeddings.size(0), device=token_embeddings.device)
            outs.append(token_embeddings[bidx, last_idx])

        if self.use_mean:
            mask = attention_mask.unsqueeze(-1).to(token_embeddings.dtype)
            summed = (token_embeddings * mask).sum(dim=1)
            denom  = mask.sum(dim=1).clamp(min=1e-6)
            outs.append(summed / denom)

        if self.use_max:
            minus_inf = torch.finfo(token_embeddings.dtype).min
            masked = token_embeddings.masked_fill(attention_mask.eq(0).unsqueeze(-1), minus_inf)
            outs.append(masked.max(dim=1).values)

        if not outs:
            raise ValueError("At least one pooling must be enabled.")

        sentence_embedding = torch.cat(outs, dim=1) if len(outs) > 1 else outs[0]
        features["sentence_embedding"] = sentence_embedding
        return features

    def get_sentence_embedding_dimension(self):
        mult = int(self.use_first) + int(self.use_last) + int(self.use_mean) + int(self.use_max)
        return self.word_embedding_dimension * mult


# =========================
# 2) pooling 조건 맵
# =========================
CFG_MAP = {
    "C":  dict(first=True,  last=False, mean=False, max=False),
    "X":  dict(first=False, last=False, mean=False, max=True),
    "M":  dict(first=False, last=False, mean=True,  max=False),
    "CM": dict(first=True,  last=False, mean=True,  max=False),
    "CX": dict(first=True,  last=False, mean=False, max=True),
    "XM": dict(first=False, last=False, mean=True,  max=True),
}


# =========================
# 3) 컬렉션 prefix -> 모델 레지스트리
# =========================
MODEL_REGISTRY = {
    "SBERT": dict(model_name="sentence-transformers/all-MiniLM-L6-v2", is_decoder=False, force_append_eos=False),
    "bge":   dict(model_name="BAAI/bge-m3",                           is_decoder=False, force_append_eos=False),
    "e5":  dict(model_name="intfloat/multilingual-e5-large",                       is_decoder=False,  force_append_eos=False),
    "llama": dict(model_name="meta-llama/Llama-3.2-1B",               is_decoder=True,  force_append_eos=True),
   "bling": dict(model_name="Lajavaness/bilingual-embedding-large",               is_decoder=False,  force_append_eos = False),
  "solon": dict(model_name="OrdalieTech/Solon-embeddings-large-0.1",               is_decoder=False,  force_append_eos=False),
  "mxbai": dict(model_name="mixedbread-ai/mxbai-embed-large-v1",               is_decoder=False,  force_append_eos=False),
  "gemma": dict(model_name="google/embeddinggemma-300m",               is_decoder=False,  force_append_eos=False)
   
}


# =========================
# 4) SentenceTransformer/Wrapper 캐시
# =========================
_model_cache = {}
_wrapper_cache = {}


def parse_collection(collection_name: str):
    if "_" not in collection_name:
        raise ValueError(f"collection_name must contain '_' like 'bge_C': {collection_name}")
    prefix, cond = collection_name.rsplit("_", 1)
    if cond not in CFG_MAP:
        raise ValueError(f"Unknown pooling condition suffix={cond} in {collection_name}")
    if prefix not in MODEL_REGISTRY:
        raise ValueError(f"Unknown collection prefix={prefix} in {collection_name} (add to MODEL_REGISTRY)")
    return prefix, cond


def build_sentence_transformer(prefix: str, cond: str, device: str):
    key = (prefix, cond, device)
    if key in _model_cache:
        return _model_cache[key]

    info = MODEL_REGISTRY[prefix]
    model_name = info["model_name"]
    is_decoder = info["is_decoder"]

    model_args = {
        "trust_remote_code": True,
        "torch_dtype": torch.float16 if device == "cuda" else torch.float32,
    }
   
    word_embedding_model = models.Transformer(
        model_name_or_path=model_name,
        model_args=model_args,
        
    )

    tokenizer = word_embedding_model.tokenizer
    auto_model = word_embedding_model.auto_model

    if tokenizer.pad_token_id is None:
        if tokenizer.eos_token_id is not None:
            tokenizer.pad_token = tokenizer.eos_token
        else:
            tokenizer.add_special_tokens({"pad_token": "[PAD]"})
            auto_model.resize_token_embeddings(len(tokenizer))

    auto_model.config.pad_token_id = tokenizer.pad_token_id

    cfg = CFG_MAP[cond].copy()

    if is_decoder and cfg["first"]:
        cfg["first"] = False
        cfg["last"] = True

    pooling_model = UnifiedConcatPooling(
        word_embedding_dimension=word_embedding_model.get_word_embedding_dimension(),
        use_first=cfg["first"],
        use_last=cfg["last"],
        use_mean=cfg["mean"],
        use_max=cfg["max"],
    )

    st = SentenceTransformer(modules=[word_embedding_model, pooling_model]).to(device)
    _model_cache[key] = (st, tokenizer, info["force_append_eos"])
    return _model_cache[key]


def make_wrapper_for_collection(collection_name: str, device: str) -> Embeddings:
    if collection_name in _wrapper_cache:
        return _wrapper_cache[collection_name]

    prefix, cond = parse_collection(collection_name)
    st, tokenizer, force_append_eos = build_sentence_transformer(prefix, cond, device)

    EOS = tokenizer.eos_token or ""

    def maybe_append_eos(text: str) -> str:
        if not force_append_eos or not EOS:
            return text
        text = "" if text is None else str(text)
        return text if text.endswith(EOS) else (text + EOS)

    class MyCustomModelWrapper(Embeddings):
        def __init__(self, model: SentenceTransformer, batch_size: int = 32, normalize: bool = False):
            self.model = model
            self.batch_size = batch_size
            self.normalize = normalize
            self.model.eval()

        def embed_documents(self, texts):
            texts = [maybe_append_eos(t) for t in texts]
            vecs = self.model.encode(
                texts, batch_size=self.batch_size,
                convert_to_numpy=True,
                normalize_embeddings=self.normalize,
                show_progress_bar=False,
            )
            return vecs.tolist()

        def embed_query(self, text):
            text = maybe_append_eos(text)
            vec = self.model.encode(
                [text],
                convert_to_numpy=True,
                normalize_embeddings=self.normalize,
                show_progress_bar=False,
            )[0]
            return vec.tolist()

    wrapper = MyCustomModelWrapper(st, batch_size=32, normalize=False)
    _wrapper_cache[collection_name] = wrapper
    return wrapper


# =========================
# 5) YAML prompt loader + ✅ [SYSTEM]/[User] 분리
# =========================
def load_prompt_text_from_yaml(path: str) -> str:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Prompt file not found: {path}")

    raw = open(path, "r", encoding="utf-8").read()

    try:
        data = yaml.safe_load(raw)
    except Exception:
        return raw

    if isinstance(data, dict):
        for key in ["template", "prompt", "text", "content"]:
            if key in data and isinstance(data[key], str) and data[key].strip():
                return data[key]

        if "messages" in data and isinstance(data["messages"], list):
            parts = []
            for m in data["messages"]:
                if isinstance(m, dict):
                    c = m.get("content") or m.get("template") or ""
                    if isinstance(c, str) and c.strip():
                        parts.append(c.strip())
            if parts:
                return "\n\n".join(parts)

    if isinstance(data, list):
        parts = []
        for item in data:
            if isinstance(item, str) and item.strip():
                parts.append(item.strip())
            elif isinstance(item, dict):
                c = item.get("content") or item.get("template") or ""
                if isinstance(c, str) and c.strip():
                    parts.append(c.strip())
        if parts:
            return "\n\n".join(parts)

    return raw


def split_system_user(prompt_text: str):
    """
    prompt_text 안에 [SYSTEM] ... [User]/[USER] ... 가 있으면 분리.
    없으면 system="" / user=prompt_text 로 둔다.
    """
    t = prompt_text or ""
    if "[SYSTEM]" in t and "[User]" in t:
        sys_part = t.split("[SYSTEM]", 1)[1].split("[User]", 1)[0].strip()
        user_part = t.split("[User]", 1)[1].strip()
        return sys_part, user_part
    if "[SYSTEM]" in t and "[USER]" in t:
        sys_part = t.split("[SYSTEM]", 1)[1].split("[USER]", 1)[0].strip()
        user_part = t.split("[USER]", 1)[1].strip()
        return sys_part, user_part
    return "", t.strip()


# =========================
# 6) Evaluator
# =========================
class RAGEvaluator:
    def __init__(
        self,
        dataset_csv: str,
        collection_name: str,
        k: int,
        embedding_function: Embeddings,
        chroma_dir: str,
        llm_model: str = LLM_MODEL,
        prompt_path: str = PROMPT_PATH,
        do_llm: bool = True,
        save_contexts: bool = True,
    ):
        self.dataset_csv = dataset_csv
        self.collection_name = collection_name
        self.k = k
        self.embedding_function = embedding_function
        self.chroma_dir = chroma_dir
        self.llm_model = llm_model
        self.prompt_path = prompt_path
        self.do_llm = do_llm
        self.save_contexts = save_contexts

        dataset_name = os.path.splitext(os.path.basename(dataset_csv))[0]
        self.run_name = f"{dataset_name}__{self.collection_name}_k{self.k}"

        self.result_csv_path = os.path.join(RESULTS_DIR, self.run_name + ".csv")
        self.eval_csv_path = os.path.join(SUMMARY_DIR, self.run_name + "_retrieval.csv")
        self.debug_csv_path = os.path.join(SUMMARY_DIR, self.run_name + "_debug_retrieval.csv")

        self.df_questions = None
        self.df_results = None
        self.df_eval = None

        self._init_vectorstore()
        if self.do_llm:
            self._init_llm_and_prompt()

    # ---------- utils ----------
    @staticmethod
    def _maybe_parse(x):
        if isinstance(x, str):
            s = x.strip()
            if (s.startswith("[") and s.endswith("]")) or (s.startswith("{") and s.endswith("}")):
                try:
                    return ast.literal_eval(s)
                except Exception:
                    return x
        return x

    @staticmethod
    def _normalize(text):
        if text is None:
            return ""
        return unicodedata.normalize("NFC", str(text)).strip()

    @staticmethod
    def _unwrap_singleton(x, max_depth=3):
        # [[...]] 같은 “중첩 singleton list” 풀기
        for _ in range(max_depth):
            if isinstance(x, list) and len(x) == 1:
                x = x[0]
            else:
                break
        return x

    @staticmethod
    def _extract_pmid(value):
        if value is None:
            return None
        if isinstance(value, int):
            return str(value)
        if isinstance(value, dict):
            if "pmid" in value:
                return RAGEvaluator._extract_pmid(value["pmid"])
            if "document" in value:
                return RAGEvaluator._extract_pmid(value["document"])

        s = str(value).strip()
        if s.isdigit():
            return s

        patterns = [
            r"ncbi\.nlm\.nih\.gov/pubmed/(\d+)",
            r"www\.ncbi\.nlm\.nih\.gov/pubmed/(\d+)",
            r"pubmed\.ncbi\.nlm\.nih\.gov/(\d+)",
            r"/pubmed/(\d+)",
        ]
        for pat in patterns:
            m = re.search(pat, s)
            if m:
                return m.group(1)

        m = re.search(r"(\d+)$", s)
        if m and ("pubmed" in s or "ncbi" in s):
            return m.group(1)

        return None

    def _norm_pmid(self, pmid, source=None):
        p = self._extract_pmid(pmid)
        if not p and source:
            p = self._extract_pmid(source)
        return p

    @staticmethod
    def _safe_json(text: str):
        if text is None:
            return None
        t = text.strip()
        try:
            return json.loads(t)
        except Exception:
            pass
        m = re.search(r"\{.*\}", t, flags=re.DOTALL)
        if not m:
            return None
        try:
            return json.loads(m.group(0))
        except Exception:
            return None

    @staticmethod
    def _find_offsets(full_text: str, span_text: str):
        if not full_text or not span_text:
            return None, None
        idx = full_text.find(span_text)
        if idx == -1:
            return None, None
        return idx, idx + len(span_text)

    # ✅ pred exact를 "gold와 같은 형태"로 맞춰 저장
    @staticmethod
    def _to_gold_format(qtype: str, pred_exact):
        qtype = (qtype or "").lower().strip()

        if qtype == "factoid":
            p = "" if pred_exact is None else str(pred_exact).strip()
            return [[p]] if p else []

        if qtype == "list":
            if not pred_exact:
                return []
            return [[str(x).strip()] for x in pred_exact if str(x).strip()]

        if qtype == "yesno":
            p = "" if pred_exact is None else str(pred_exact).strip().lower()
            return p if p in ["yes", "no"] else ""

        return "" if pred_exact is None else str(pred_exact).strip()

    # ---------- gold/pred helpers (BioASQ exact_answer 형태 대응) ----------
    @staticmethod
    def _gold_groups(exact_answer_field):
        x = RAGEvaluator._maybe_parse(exact_answer_field)
        groups = []

        if isinstance(x, list):
            for item in x:
                if isinstance(item, list):
                    s = {
                        RAGEvaluator._normalize(v).lower()
                        for v in item
                        if v is not None and RAGEvaluator._normalize(v)
                    }
                    if s:
                        groups.append(s)
                else:
                    v = RAGEvaluator._normalize(item).lower()
                    if v:
                        groups.append({v})
        elif isinstance(x, str):
            v = RAGEvaluator._normalize(x).lower()
            if v:
                groups.append({v})

        return groups

    @staticmethod
    def _pred_items(pred_answer):
        if pred_answer is None:
            return []
        if isinstance(pred_answer, list):
            return [RAGEvaluator._normalize(v).lower() for v in pred_answer if RAGEvaluator._normalize(v)]
        if isinstance(pred_answer, str):
            s = RAGEvaluator._normalize(pred_answer)
            if not s:
                return []
            parts = re.split(r"[;\n,]\s*", s)
            return [RAGEvaluator._normalize(p).lower() for p in parts if RAGEvaluator._normalize(p)]
        return []

    # list: group EM/F1
    def _list_group_f1(self, pred_answer, exact_answer_field):
        gold_groups = self._gold_groups(exact_answer_field)
        pred_items = self._pred_items(pred_answer)

        if not gold_groups and not pred_items:
            return 1.0
        if not gold_groups or not pred_items:
            return 0.0

        matched = 0
        used_pred = set()

        for g in gold_groups:
            hit = False
            for i, p in enumerate(pred_items):
                if i in used_pred:
                    continue
                if p in g:
                    hit = True
                    used_pred.add(i)
                    break
            if hit:
                matched += 1

        tp = matched
        precision = tp / len(pred_items)
        recall = tp / len(gold_groups)
        return 0.0 if (precision + recall) == 0 else 2 * precision * recall / (precision + recall)

    def _list_group_em(self, pred_answer, exact_answer_field):
        gold_groups = self._gold_groups(exact_answer_field)
        pred_items = self._pred_items(pred_answer)

        if not gold_groups and not pred_items:
            return 1
        if not gold_groups or not pred_items:
            return 0

        matched = 0
        used_pred = set()

        for g in gold_groups:
            hit = False
            for i, p in enumerate(pred_items):
                if i in used_pred:
                    continue
                if p in g:
                    hit = True
                    used_pred.add(i)
                    break
            if hit:
                matched += 1

        if matched == len(gold_groups) and len(used_pred) == len(pred_items):
            return 1
        return 0

    # ---------- init vectorstore ----------
    def _init_vectorstore(self):
        self.vectorstore = Chroma(
            collection_name=self.collection_name,
            persist_directory=self.chroma_dir,
            embedding_function=self.embedding_function,
        )
        self.retriever = self.vectorstore.as_retriever(
            search_type="similarity",
            search_kwargs={"k": self.k},
        )
        print(f"✅ Chroma 로드 완료: {self.collection_name} (k={self.k})")

    # ---------- init llm + prompt ----------
    def _init_llm_and_prompt(self):
        print(f"--- LLM 로드: ChatOllama({self.llm_model}) ---")
        self.llm = ChatOllama(model=self.llm_model)
        print("✅ LLM 로드 완료")

        print(f"--- Prompt YAML 로드: {self.prompt_path} ---")
        self.prompt_text = load_prompt_text_from_yaml(self.prompt_path)
        if not isinstance(self.prompt_text, str) or not self.prompt_text.strip():
            raise ValueError("Prompt YAML에서 유효한 프롬프트 텍스트를 찾지 못했습니다.")

        # ✅ [SYSTEM]/[User] 분리 (없으면 user로 취급)
        self.system_tmpl, self.user_tmpl = split_system_user(self.prompt_text)

        # user 파트가 아예 없으면, 최소 user 템플릿을 깔아줌
        if not self.user_tmpl.strip():
            self.user_tmpl = "[Question Type] question: {question}\nAnswer:"

        print("✅ Prompt 로드 완료 (system/user split)")

    # ---------- load ----------
    def load_questions(self):
        self.df_questions = pd.read_csv(self.dataset_csv)

        # ✅ snippets는 안 쓰므로 파싱/필수 컬럼에서 제거
        for col in ["documents", "exact_answer"]:
            if col in self.df_questions.columns:
                self.df_questions[col] = self.df_questions[col].apply(self._maybe_parse)

        need_cols = ["body", "documents", "type", "id", "exact_answer"]
        missing = [c for c in need_cols if c not in self.df_questions.columns]
        if missing:
            raise ValueError(f"질문 CSV에 필요한 컬럼이 없음: {missing}")

        if "ideal_answer" not in self.df_questions.columns:
            self.df_questions["ideal_answer"] = ""

        print(f"✅ 질문 로드 완료: {len(self.df_questions)} rows")

    # ---------- context builder ----------
    def _build_context(self, retrieved_docs):
        blocks = []
        docid_to_text = {}
        docid_to_pmid = {}
        total = 0

        for rank, doc in enumerate(retrieved_docs):
            pmid = self._norm_pmid(doc.metadata.get("pmid"), doc.metadata.get("source")) or "UNKNOWN"

            chunk_text = (doc.page_content or "").strip()
            if len(chunk_text) > MAX_DOC_CHARS:
                chunk_text = chunk_text[:MAX_DOC_CHARS]

            doc_id = f"{pmid}#{rank}"
            block = f"[DOCID={doc_id}][PMID={pmid}]\n{chunk_text}\n"

            if total + len(block) > MAX_TOTAL_CHARS:
                break

            blocks.append(block)
            total += len(block)

            docid_to_text[doc_id] = chunk_text
            docid_to_pmid[doc_id] = pmid

        return "\n".join(blocks), docid_to_text, docid_to_pmid

    # ✅ system/user 각각에 치환
    def _format_tmpl(self, tmpl: str, context_str: str, question: str, qtype: str) -> str:
        p = tmpl or ""
        if "{context}" in p:
            p = p.replace("{context}", context_str)
        if "{question}" in p:
            p = p.replace("{question}", question)
        if "{type}" in p:
            p = p.replace("{type}", qtype)
        return p

    # ---------- parse LLM output ----------
    def _parse_llm_output(self, raw_text: str):
        parsed = self._safe_json(raw_text)
        pred_exact = ""
        pred_ideal = ""
        sources = []

        if isinstance(parsed, dict):
            pred_exact = parsed.get("exact_answer", "")
            if pred_exact == "" and "answer" in parsed:
                pred_exact = parsed.get("answer", "")

            pred_ideal = parsed.get("ideal_answer", "")

            src = parsed.get("sources", []) or []
            if isinstance(src, list):
                sources = src

            return pred_exact, pred_ideal, sources

        t = (raw_text or "").strip()
        return (t if t else ""), "", []

    # ✅ 타입별 exact sanitize (factoid/list에서 yes/no 방지)
    def _sanitize_pred_exact_by_type(self, qtype: str, pred_exact):
        qtype = (qtype or "").lower().strip()

        # 중첩 singleton 풀기
        pred_exact = self._unwrap_singleton(pred_exact)

        # 공통: 문자열이면 strip
        if isinstance(pred_exact, str):
            s = pred_exact.strip()
        else:
            s = None

        # ✅ factoid인데 yes/no면 무조건 빈값
        if qtype == "factoid":
            if isinstance(pred_exact, list):
                pred_exact = pred_exact[0] if pred_exact else ""
                pred_exact = "" if pred_exact is None else str(pred_exact).strip()
            else:
                pred_exact = "" if pred_exact is None else str(pred_exact).strip()

            if pred_exact.lower() in ["yes", "no"]:
                pred_exact = ""  # 핵심 가드
            return pred_exact

        # ✅ list인데 yes/no면 []
        if qtype == "list":
            if pred_exact is None:
                return []
            if isinstance(pred_exact, str):
                if pred_exact.strip().lower() in ["yes", "no"]:
                    return []
            # 아래에서 list 형태로 강제
            if isinstance(pred_exact, str):
                s = pred_exact.strip()
                if s == "":
                    return []
                if s.startswith("[") and s.endswith("]"):
                    try:
                        tmp = json.loads(s)
                        if isinstance(tmp, list):
                            return tmp
                    except Exception:
                        pass
                return [a for a in re.split(r"[;\n,]\s*", s) if a]
            if isinstance(pred_exact, list):
                # 리스트 내부도 정리
                out = []
                for v in pred_exact:
                    vv = "" if v is None else str(v).strip()
                    if vv:
                        out.append(vv)
                return out
            return [str(pred_exact).strip()] if str(pred_exact).strip() else []

        # yesno
        if qtype == "yesno":
            if isinstance(pred_exact, list):
                pred_exact = pred_exact[0] if pred_exact else ""
            pred_exact = "" if pred_exact is None else str(pred_exact).strip().lower()
            return pred_exact if pred_exact in ["yes", "no"] else ""

        # 기타
        if isinstance(pred_exact, list):
            pred_exact = pred_exact[0] if pred_exact else ""
        return "" if pred_exact is None else str(pred_exact).strip()

    # ---------- generation ----------
    def run_generation(self):
        if self.df_questions is None:
            self.load_questions()

        results = []
        debug_rows = []
        start = time.time()

        for idx, (_, row) in enumerate(tqdm(self.df_questions.iterrows(), total=len(self.df_questions), desc="Retrieve+Generate")):
            question = self._normalize(row.get("body", ""))
            qtype = self._normalize(row.get("type", "")).lower()

            gold_exact = row.get("exact_answer")
            gold_ideal = row.get("ideal_answer", "")

            # 1) retrieve
            retrieved_docs = self.retriever.invoke(question)

            retrieved_pmids = []
            retrieved_docids = []
            contexts = []

            for rank, doc in enumerate(retrieved_docs):
                pmid = self._norm_pmid(doc.metadata.get("pmid"), doc.metadata.get("source")) or "UNKNOWN"
                retrieved_pmids.append(pmid)
                retrieved_docids.append(f"{pmid}#{rank}")
                if self.save_contexts:
                    contexts.append(doc.page_content)

            # 2) generation
            pred_exact_answer_goldfmt = []
            pred_ideal_answer = ""
            sources_out = []
            raw = ""

            # 내부 평가용
            generated_answer = None
            ideal_answer = ""

            if self.do_llm:
                context_str, docid_to_text, docid_to_pmid = self._build_context(retrieved_docs)

                # ✅ system/user 각각 치환 후 messages로 invoke
                sys_msg = self._format_tmpl(self.system_tmpl, context_str, question, qtype)
                usr_msg = self._format_tmpl(self.user_tmpl, context_str, question, qtype)

                try:
                    msg = self.llm.invoke([
                        SystemMessage(content=sys_msg),
                        HumanMessage(content=usr_msg),
                    ])
                    raw = msg.content if hasattr(msg, "content") else str(msg)
                except Exception as e:
                    raw = ""
                    print(f"\n[WARN] LLM invoke error: {repr(e)}")

                pred_exact, pred_ideal, sources_in = self._parse_llm_output(raw)

                # ✅ 타입별 sanitize (factoid에서 yes/no 차단 포함)
                pred_exact = self._sanitize_pred_exact_by_type(qtype, pred_exact)
                pred_ideal = "" if pred_ideal is None else str(pred_ideal).strip()

                generated_answer = pred_exact
                ideal_answer = pred_ideal

                # ✅ 저장용: gold 형태로
                pred_exact_answer_goldfmt = self._to_gold_format(qtype, pred_exact)
                pred_ideal_answer = pred_ideal

                # ---------- sources 검증(있을 때만) ----------
                if isinstance(sources_in, list):
                    for s in sources_in:
                        if not isinstance(s, dict):
                            continue

                        doc_id = s.get("doc_id")
                        pmid = s.get("pmid")
                        quote = s.get("quote")

                        if quote is None and isinstance(s.get("text"), str):
                            quote = s.get("text")

                        doc_id = str(doc_id) if doc_id is not None else None
                        pmid = str(pmid) if pmid is not None else None
                        quote = quote if isinstance(quote, str) else None

                        if not doc_id or doc_id not in docid_to_text:
                            continue

                        chunk_text = docid_to_text.get(doc_id, "")
                        pmid_expected = docid_to_pmid.get(doc_id)

                        pmid_norm = self._extract_pmid(pmid) if pmid else None
                        pmid_expected_norm = self._extract_pmid(pmid_expected) if pmid_expected else None

                        if not pmid_norm:
                            pmid_norm = pmid_expected_norm
                        if pmid_expected_norm and pmid_norm != pmid_expected_norm:
                            continue

                        if quote is None:
                            continue

                        b, e = self._find_offsets(chunk_text, quote)
                        if b is None or e is None:
                            continue

                        sources_out.append({
                            "doc_id": doc_id,
                            "pmid": pmid_norm,
                            "text": quote,
                            "offsetInBeginSection": b,
                            "offsetInEndSection": e,
                        })

                if EVIDENCE_FALLBACK_TO_RETRIEVAL and not sources_out:
                    sources_out = [
                        {
                            "doc_id": f"{p}#{i}",
                            "pmid": p,
                            "text": None,
                            "offsetInBeginSection": None,
                            "offsetInEndSection": None,
                        }
                        for i, p in enumerate(retrieved_pmids)
                        if p and p != "UNKNOWN"
                    ]

            # 3) answer metrics (임시/참고용)
            if qtype == "list":
                answer_em = float(self._list_group_em(generated_answer, gold_exact))
                answer_f1 = float(self._list_group_f1(generated_answer, gold_exact))

            elif qtype == "factoid":
                gold_groups = self._gold_groups(gold_exact)
                pred = self._normalize(generated_answer).lower()
                hit = 0.0
                if pred and gold_groups:
                    for g in gold_groups:
                        if pred in g:
                            hit = 1.0
                            break
                answer_em = hit
                answer_f1 = hit

            elif qtype == "yesno":
                gold = self._normalize(gold_exact).lower()
                pred = self._normalize(generated_answer).lower()
                hit = 1.0 if (gold in ["yes", "no"] and pred == gold) else 0.0
                answer_em = hit
                answer_f1 = hit

            else:
                answer_em = np.nan
                answer_f1 = np.nan

            out = {
                "body": row.get("body"),
                "documents": row.get("documents"),
                "type": row.get("type"),
                "id": row.get("id"),

                # gold
                "gold_exact_answer": gold_exact,
                "gold_ideal_answer": gold_ideal,

                # retrieval
                "retrieved_pmids": retrieved_pmids,
                "retrieved_docids": retrieved_docids,

                # pred
                "pred_exact_answer": pred_exact_answer_goldfmt,  # ✅ 하나만 저장(골드형태)
                "pred_ideal_answer": pred_ideal_answer,

                "raw_llm": raw,
                "sources": sources_out,

                "answer_em": float(answer_em) if answer_em is not np.nan else np.nan,
                "answer_f1": float(answer_f1) if answer_f1 is not np.nan else np.nan,
            }
            if self.save_contexts:
                out["contexts"] = contexts

            results.append(out)

            if DEBUG_SAVE_FIRST_N and idx < DEBUG_SAVE_FIRST_N:
                gold_docs = row.get("documents", []) or []
                gold_pmids = [self._extract_pmid(d) for d in gold_docs]
                gold_pmids = [p for p in gold_pmids if p]

                overlap = set(gold_pmids) & set([p for p in retrieved_pmids if p and p != "UNKNOWN"])
                unk_ratio = sum(p == "UNKNOWN" for p in retrieved_pmids) / max(len(retrieved_pmids), 1)

                debug_rows.append({
                    "qid": row.get("id"),
                    "qtype": qtype,
                    "question_head": question[:120],
                    "gold_count": len(set(gold_pmids)),
                    "retrieved_count": len(set(retrieved_pmids)),
                    "overlap_count": len(overlap),
                    "overlap_example": ";".join(list(overlap)[:5]),
                    "gold_pmids_head": ";".join(gold_pmids[:10]),
                    "retrieved_pmids_head": ";".join([p for p in retrieved_pmids[:10]]),
                    "unknown_ratio": float(unk_ratio),
                })

        self.df_results = pd.DataFrame(results)
        self.df_results.to_csv(self.result_csv_path, index=False, encoding="utf-8-sig")
        print(f"📁 결과 저장: {self.result_csv_path} (time={time.time()-start:.1f}s)")

        if debug_rows:
            pd.DataFrame(debug_rows).to_csv(self.debug_csv_path, index=False, encoding="utf-8-sig")
            print(f"📁 debug 저장: {self.debug_csv_path}")

    # ---------- evidence doc-level metrics ----------
    def _evidence_doc_scores(self, gold_documents, pred_sources):
        gold = set()
        for d in (gold_documents or []):
            pmid = self._extract_pmid(d)
            if pmid:
                gold.add(pmid)

        pred = set()
        for s in (pred_sources or []):
            if isinstance(s, dict) and s.get("pmid"):
                p = self._extract_pmid(s.get("pmid"))
                if p:
                    pred.add(p)

        if not gold:
            return np.nan, np.nan, np.nan

        inter = gold & pred
        prec = len(inter) / max(len(pred), 1)
        rec = len(inter) / len(gold)
        hit = 1.0 if len(inter) > 0 else 0.0
        return prec, rec, hit

    # ---------- retrieval evaluation ----------
    def run_retrieval_eval(self):
        if self.df_results is None:
            self.df_results = pd.read_csv(self.result_csv_path)

        df = self.df_results.copy()

        for c in ["documents", "retrieved_pmids", "retrieved_docids",
                  "gold_exact_answer", "gold_ideal_answer",
                  "pred_exact_answer", "pred_ideal_answer",
                  "sources", "contexts"]:
            if c in df.columns:
                df[c] = df[c].apply(self._maybe_parse)

        eval_rows = []
        for _, row in df.iterrows():
            gold_set = set()
            docs = row.get("documents", None)
            if isinstance(docs, list):
                for d in docs:
                    pmid = self._extract_pmid(d)
                    if pmid:
                        gold_set.add(pmid)

            pred = row.get("retrieved_pmids", [])
            pred_norm = []
            if isinstance(pred, list):
                seen = set()
                for p in pred:
                    pm = self._extract_pmid(p) or (str(p).strip() if p is not None else None)
                    if not pm or pm == "UNKNOWN":
                        continue
                    if pm not in seen:
                        pred_norm.append(pm)
                        seen.add(pm)

            L = min(self.k, len(pred_norm))
            pred_top = pred_norm[:L]

            if len(gold_set) == 0 or L == 0:
                recall_k = np.nan
                precision_k = np.nan
                hit_k = np.nan
                ndcg_k = np.nan
            else:
                hit_count = sum([1 for p in pred_top if p in gold_set])
                recall_k = hit_count / len(gold_set)
                precision_k = hit_count / L
                hit_k = 1.0 if hit_count > 0 else 0.0

                rel = [1 if (p in gold_set) else 0 for p in pred_top]
                if sum(rel) == 0:
                    ndcg_k = 0.0
                else:
                    L2 = len(rel)
                    K2 = max(self.k, 2)
                    rel_pad = (rel + [0.0] * (K2 - L2))[:K2]
                    score_pad = (list(range(L2, 0, -1)) + [0.0] * (K2 - L2))[:K2]
                    y_true  = np.asarray([rel_pad], dtype=float)
                    y_score = np.asarray([score_pad], dtype=float)
                    ndcg_k = float(ndcg_score(y_true, y_score, k=min(self.k, K2)))

            ev_prec, ev_rec, ev_hit = self._evidence_doc_scores(
                row.get("documents", []),
                row.get("sources", []),
            )

            eval_rows.append({
                "id": row.get("id"),
                "type": row.get("type"),
                "recall@k": recall_k,
                "precision@k": precision_k,
                "ndcg@k": ndcg_k,
                "hit@k": hit_k,

                "answer_em": row.get("answer_em", np.nan),
                "answer_f1": row.get("answer_f1", np.nan),

                "evidence_precision": ev_prec,
                "evidence_recall": ev_rec,
                "evidence_hit": ev_hit,

                "gold_doc_count": len(gold_set),
            })

        self.df_eval = pd.DataFrame(eval_rows)
        self.df_eval.to_csv(self.eval_csv_path, index=False, encoding="utf-8-sig")
        print(f"📁 retrieval 평가 저장: {self.eval_csv_path}")

        overall_recall = float(self.df_eval["recall@k"].mean(skipna=True))
        overall_precision = float(self.df_eval["precision@k"].mean(skipna=True))
        overall_ndcg = float(self.df_eval["ndcg@k"].mean(skipna=True))
        overall_hit = float(self.df_eval["hit@k"].mean(skipna=True))

        overall_answer_em = float(self.df_eval["answer_em"].mean(skipna=True))
        overall_answer_f1 = float(self.df_eval["answer_f1"].mean(skipna=True))

        overall_ev_precision = float(self.df_eval["evidence_precision"].mean(skipna=True))
        overall_ev_recall = float(self.df_eval["evidence_recall"].mean(skipna=True))
        overall_ev_hit = float(self.df_eval["evidence_hit"].mean(skipna=True))

        print(f"- Recall@K          : {overall_recall:.6f}")
        print(f"- Precision@K       : {overall_precision:.6f}")
        print(f"- NDCG@K            : {overall_ndcg:.6f}")
        print(f"- Hit@K             : {overall_hit:.6f}")
        print(f"- AnswerEM          : {overall_answer_em:.6f}")
        print(f"- AnswerF1          : {overall_answer_f1:.6f}")
        print(f"- EvidencePrecision : {overall_ev_precision:.6f}")
        print(f"- EvidenceRecall    : {overall_ev_recall:.6f}")
        print(f"- EvidenceHit       : {overall_ev_hit:.6f}")

    def run_all(self):
        self.run_generation()
        self.run_retrieval_eval()


# =========================
# 7) main
# =========================
if __name__ == "__main__":
    for col in collection_list:
        wrapper = make_wrapper_for_collection(col, device=DEVICE)

        for k in k_list:
            print(f"\n=== Running | dataset={os.path.basename(DATASET_LIST_CSV)} | collection={col} | k={k} ===")
            evaluator = RAGEvaluator(
                dataset_csv=DATASET_LIST_CSV,
                collection_name=col,
                k=k,
                embedding_function=wrapper,
                chroma_dir=CHROMA_DIR,
                do_llm=True,
                save_contexts=True,
            )
            evaluator.run_all()
            print("****************************************\n")



=== Running | dataset=bioasq_factoid.csv | collection=gemma_C | k=5 ===
✅ Chroma 로드 완료: gemma_C (k=5)
--- LLM 로드: ChatOllama(gemma3:4b) ---
✅ LLM 로드 완료
--- Prompt YAML 로드: C:/Users/USER/Desktop/RAG_Korean/prompt/rag_prompt2.yaml ---
✅ Prompt 로드 완료 (system/user split)
✅ 질문 로드 완료: 327 rows


Retrieve+Generate: 100%|██████████| 327/327 [16:55<00:00,  3.11s/it]


📁 결과 저장: C:/Users/USER/Desktop/RAG_Korean/500-100/result\bioasq_factoid__gemma_C_k5.csv (time=1015.9s)
📁 debug 저장: C:/Users/USER/Desktop/RAG_Korean/500-100/result_summary\bioasq_factoid__gemma_C_k5_debug_retrieval.csv
📁 retrieval 평가 저장: C:/Users/USER/Desktop/RAG_Korean/500-100/result_summary\bioasq_factoid__gemma_C_k5_retrieval.csv
- Recall@K          : 0.006239
- Precision@K       : 0.016514
- NDCG@K            : 0.058894
- Hit@K             : 0.079511
- AnswerEM          : 0.006116
- AnswerF1          : 0.006116
- EvidencePrecision : 0.016514
- EvidenceRecall    : 0.006239
- EvidenceHit       : 0.079511
****************************************


=== Running | dataset=bioasq_factoid.csv | collection=gemma_C | k=10 ===
✅ Chroma 로드 완료: gemma_C (k=10)
--- LLM 로드: ChatOllama(gemma3:4b) ---
✅ LLM 로드 완료
--- Prompt YAML 로드: C:/Users/USER/Desktop/RAG_Korean/prompt/rag_prompt2.yaml ---
✅ Prompt 로드 완료 (system/user split)
✅ 질문 로드 완료: 327 rows


Retrieve+Generate: 100%|██████████| 327/327 [17:13<00:00,  3.16s/it]


📁 결과 저장: C:/Users/USER/Desktop/RAG_Korean/500-100/result\bioasq_factoid__gemma_C_k10.csv (time=1033.7s)
📁 debug 저장: C:/Users/USER/Desktop/RAG_Korean/500-100/result_summary\bioasq_factoid__gemma_C_k10_debug_retrieval.csv
📁 retrieval 평가 저장: C:/Users/USER/Desktop/RAG_Korean/500-100/result_summary\bioasq_factoid__gemma_C_k10_retrieval.csv
- Recall@K          : 0.008054
- Precision@K       : 0.011009
- NDCG@K            : 0.065294
- Hit@K             : 0.100917
- AnswerEM          : 0.009174
- AnswerF1          : 0.009174
- EvidencePrecision : 0.011009
- EvidenceRecall    : 0.008054
- EvidenceHit       : 0.100917
****************************************


=== Running | dataset=bioasq_factoid.csv | collection=gemma_C | k=20 ===
✅ Chroma 로드 완료: gemma_C (k=20)
--- LLM 로드: ChatOllama(gemma3:4b) ---
✅ LLM 로드 완료
--- Prompt YAML 로드: C:/Users/USER/Desktop/RAG_Korean/prompt/rag_prompt2.yaml ---
✅ Prompt 로드 완료 (system/user split)
✅ 질문 로드 완료: 327 rows


...

📁 결과 저장: C:/Users/USER/Desktop/RAG_Korean/500-100/result\bioasq_factoid__gemma_M_k5.csv (time=1165.8s)
📁 debug 저장: C:/Users/USER/Desktop/RAG_Korean/500-100/result_summary\bioasq_factoid__gemma_M_k5_debug_retrieval.csv
📁 retrieval 평가 저장: C:/Users/USER/Desktop/RAG_Korean/500-100/result_summary\bioasq_factoid__gemma_M_k5_retrieval.csv
- Recall@K          : 0.296265
- Precision@K       : 0.604179
- NDCG@K            : 0.783982
- Hit@K             : 0.859327
- AnswerEM          : 0.137615
- AnswerF1          : 0.137615
- EvidencePrecision : 0.604179
- EvidenceRecall    : 0.296265
- EvidenceHit       : 0.859327
****************************************


=== Running | dataset=bioasq_factoid.csv | collection=gemma_M | k=10 ===
✅ Chroma 로드 완료: gemma_M (k=10)
--- LLM 로드: ChatOllama(gemma3:4b) ---
✅ LLM 로드 완료
--- Prompt YAML 로드: C:/Users/USER/Desktop/RAG_Korean/prompt/rag_prompt2.yaml ---
✅ Prompt 로드 완료 (system/user split)
✅ 질문 로드 완료: 327 rows


Retrieve+Generate: 100%|██████████| 327/327 [19:38<00:00,  3.60s/it]


📁 결과 저장: C:/Users/USER/Desktop/RAG_Korean/500-100/result\bioasq_factoid__gemma_M_k10.csv (time=1178.3s)
📁 debug 저장: C:/Users/USER/Desktop/RAG_Korean/500-100/result_summary\bioasq_factoid__gemma_M_k10_debug_retrieval.csv
📁 retrieval 평가 저장: C:/Users/USER/Desktop/RAG_Korean/500-100/result_summary\bioasq_factoid__gemma_M_k10_retrieval.csv
- Recall@K          : 0.416006
- Precision@K       : 0.512371
- NDCG@K            : 0.786558
- Hit@K             : 0.902141
- AnswerEM          : 0.162080
- AnswerF1          : 0.162080
- EvidencePrecision : 0.512371
- EvidenceRecall    : 0.416006
- EvidenceHit       : 0.902141
****************************************


=== Running | dataset=bioasq_factoid.csv | collection=gemma_M | k=20 ===
✅ Chroma 로드 완료: gemma_M (k=20)
--- LLM 로드: ChatOllama(gemma3:4b) ---
✅ LLM 로드 완료
--- Prompt YAML 로드: C:/Users/USER/Desktop/RAG_Korean/prompt/rag_prompt2.yaml ---
✅ Prompt 로드 완료 (system/user split)
✅ 질문 로드 완료: 327 rows


Retrieve+Generate: 100%|██████████| 327/327 [21:25<00:00,  3.93s/it]


📁 결과 저장: C:/Users/USER/Desktop/RAG_Korean/500-100/result\bioasq_factoid__gemma_M_k20.csv (time=1286.0s)
📁 debug 저장: C:/Users/USER/Desktop/RAG_Korean/500-100/result_summary\bioasq_factoid__gemma_M_k20_debug_retrieval.csv
📁 retrieval 평가 저장: C:/Users/USER/Desktop/RAG_Korean/500-100/result_summary\bioasq_factoid__gemma_M_k20_retrieval.csv
- Recall@K          : 0.546406
- Precision@K       : 0.403159
- NDCG@K            : 0.779126
- Hit@K             : 0.923547
- AnswerEM          : 0.244648
- AnswerF1          : 0.244648
- EvidencePrecision : 0.403159
- EvidenceRecall    : 0.546406
- EvidenceHit       : 0.923547
****************************************



In [20]:
import gc, torch

# (선택) 네 캐시 dict가 있을 때만
try:
    _wrapper_cache.clear()
    _model_cache.clear()
except Exception:
    pass

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()  # (선택) 멀티프로세스/IPC 캐시 정리


In [ ]:
# Create Retriever
retriever = vectorstore.as_retriever(
    search_type="similarity",
    search_kwargs={"k": 3}
)

In [ ]:
!ollama list

NAME         ID              SIZE      MODIFIED    
gemma3:4b    a2af6cc3eb7f    3.3 GB    2 hours ago    


## 4. Setup RAG Pipeline

In [ ]:
# Initialize LLM
llm = ChatOllama(model="gemma3:4b")

# Pull Prompt
prompt = load_prompt("/content/drive/MyDrive/RAG_Korean/prompt/rag_prompt.yaml")

# Define Post-processing to extract source info
def format_docs(docs):
    return "\n\n".join(doc.page_content for doc in docs)

# RAG Chain
rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

/tmp/ipython-input-958517762.py:2: LangChainDeprecationWarning: The class `ChatOllama` was deprecated in LangChain 0.3.1 and will be removed in 1.0.0. An updated version of the class exists in the `langchain-ollama package and should be used instead. To use it run `pip install -U `langchain-ollama` and import as `from `langchain_ollama import ChatOllama``.
  llm = ChatOllama(model="gemma3:4b")


## 5. Evaluation Loop

In [37]:
#check result
import pandas as pd
df = pd.read_csv("C:/Users/USER/Desktop/RAG_Korean/500-100/result/bioasq_factoid__bge_CM_k20.csv")
df.head()

,body,documents,type,id,gold_exact_answer,gold_ideal_answer,retrieved_pmids,retrieved_docids,pred_exact_answer,pred_ideal_answer,raw_llm,sources,answer_em,answer_f1,contexts
0,What is the inheritance pattern of Li–Fraumeni...,"['7981072', '2190528', '9302689', '16772121', ...",factoid,52bf208003868f1b06000019,[['Autosomal dominant']],['Li-Fraumeni syndrome shows autosomal dominan...,"['9302689', '16772121', '9302689', '22672556',...","['9302689#0', '16772121#1', '9302689#2', '2267...",[['autosomal dominant']],Li-Fraumeni syndrome is characterized by autos...,"```json\n{\n ""exact_answer"": ""autosomal domin...","[{'doc_id': '9302689#0', 'pmid': '9302689', 't...",1.0,1.0,['Abstract: The Li-Fraumeni syndrome is a rare...
1,What is the main role of Ctf4 in dna replication?,"['24255107', '21470422', '20381454', '20089864...",factoid,530cf22aa177c6630c000004,[['Coordination of the progression of helicase...,['Ctf4 coordinates the progression of helicase...,"['21470422', '20089864', '21470422', '19496828...","['21470422#0', '20089864#1', '21470422#2', '19...",[['Ctf4 protein has been shown to be a central...,Ctf4 protein has been shown to be a central me...,"```json\n{\n ""exact_answer"": ""Ctf4 protein ha...","[{'doc_id': '21470422#0', 'pmid': '21470422', ...",0.0,0.0,['Title: Drosophila Ctf4 is essential for effi...
2,Which type of lung cancer is afatinib used for?,"['24086949', '23683257', '23664448', '18520300...",factoid,530cf4fe960c95ad0c00000b,"[['EGFR-mutant non small cell lung carcinoma',...",['Afatinib is a small molecule covalently bind...,"['22452896', '24086949', '23664448', '21554040...","['22452896#0', '24086949#1', '23664448#2', '21...",[['EGFR-mutant non-small cell lung cancer']],Afatinib has been extensively studied in EGFR-...,"```json\n{\n ""exact_answer"": ""EGFR-mutant non...","[{'doc_id': '22452896#0', 'pmid': '22452896', ...",0.0,0.0,['Title: Afatinib versus placebo for patients ...
3,What is the clinical indication of cardiac T1 ...,"['24124732', '24043965', '24036385', '24011774...",factoid,530cf51d5610acba0c000001,[['T1 mapping can quantitatively characterize ...,['T1 mapping can quantitatively characterize m...,"['22710483', '24566951', '23643513', '12765114...","['22710483#0', '24566951#1', '23643513#2', '12...",[['myocardial infarction']],Research into T1 mapping has largely been focu...,"```json\n{\n ""exact_answer"": ""myocardial infa...","[{'doc_id': '22710483#0', 'pmid': '22710483', ...",0.0,0.0,['Title: Assessment of cardiac involvement in ...
4,Which hormone abnormalities are characteristic...,"['23235354', '22116361', '22109890', '21511235...",factoid,53148a07dae131f847000002,[['thyroid']],['Thyroid hormone abnormalities are characteri...,"['10650967', '20501687', '18692402', '20298745...","['10650967#0', '20501687#1', '18692402#2', '20...",[['impaired iodide organification']],Impaired iodide organification is characterist...,"```json\n{\n ""exact_answer"": ""impaired iodide...","[{'doc_id': '10650967#0', 'pmid': '10650967', ...",0.0,0.0,['Abstract: Pendred syndrome is an autosomal r...


In [25]:
df['contexts'][0]

"['Title: Alteplase: new indication. Inadequately assessed in ischaemic stroke.', 'recommended as a treatment for patients with deep venous thrombosis of lower limbs and/or pelvis. Further studies are needed to define a more suitable dosage regimen of alteplase in this indication.', 'Title: Alteplase for the treatment of catheter occlusion in pediatric patients.', 'alteplase in the treatment of pulmonary thromboembolism has also been established and appears to be similar to that of streptokinase and urokinase in this indication and in arterial thrombotic occlusion. However, its use in this latter indication and in other vascular disorders has not been as extensively documented. Although trials demonstrating the efficacy of intravenous alteplase in patients with deep vein thrombosis and intra-arterial alteplase in patients with arterial thrombotic', 'trials have not established a role for alteplase in the treatment of acute coronary syndromes or deep vein thrombosis. However, alteplase 

In [26]:
# answer_em > 0 인 행만 보기
hit = df[df["answer_em"] > 0].copy()

print("EM hits:", len(hit))


EM hits: 8


In [31]:
# #check total result
import os, glob, re, ast
import numpy as np
import pandas as pd

RESULT_DIR = r"C:\Users\USER\Desktop\RAG_Korean\500-100\result"

def parse_filename(fname: str):
    base = os.path.basename(fname).strip()
    m = re.match(
        r"^bioasq_(?P<qtype>factoid|list|yesno)[_]+(?P<model>[^_]+)_(?P<pool>[^_]+)_k(?P<k>\d+)\.csv$",
        base,
        flags=re.IGNORECASE
    )
    if not m:
        return {"qtype": None, "model": None, "pool": None, "k": None}
    d = m.groupdict()
    d["qtype"] = d["qtype"].lower()
    d["k"] = int(d["k"])
    return d

def read_csv_smart(path):
    for enc in ("utf-8-sig", "utf-8", "cp949"):
        try:
            return pd.read_csv(path, encoding=enc)
        except Exception:
            pass
    return pd.read_csv(path, engine="python")

def to_list(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return []
    if isinstance(x, list):
        return x
    s = str(x).strip()
    if s == "":
        return []
    if (s.startswith("[") and s.endswith("]")) or (s.startswith("(") and s.endswith(")")):
        try:
            return to_list(ast.literal_eval(s))
        except Exception:
            pass
    return [s]

def norm_text(t):
    return re.sub(r"\s+", " ", str(t).strip().lower())

# ---------- Factoid MRR ----------
def compute_factoid_mrr(df: pd.DataFrame) -> float:
    for c in ["factoid_mrr", "mrr", "MRR", "exact_mrr"]:
        if c in df.columns:
            return float(pd.to_numeric(df[c], errors="coerce").mean())

    gold_candidates = ["gold_exact", "gold_exact_answer", "exact_answer", "gold"]
    pred_candidates = ["pred_exact", "pred_exact_answer", "pred"]

    gold_col = next((c for c in gold_candidates if c in df.columns), None)
    pred_col = next((c for c in pred_candidates if c in df.columns), None)
    if gold_col is None or pred_col is None:
        return np.nan

    mrrs = []
    for _, row in df.iterrows():
        gold_list = [norm_text(x) for x in to_list(row[gold_col])]
        pred_list = [norm_text(x) for x in to_list(row[pred_col])]
        if not pred_list:
            mrrs.append(0.0); continue

        ranks = []
        for g in gold_list:
            for i, p in enumerate(pred_list, start=1):
                if g == p:
                    ranks.append(i); break
        mrrs.append(1.0 / min(ranks) if ranks else 0.0)

    return float(np.mean(mrrs)) if mrrs else np.nan

# ---------- List F-measure ----------
def compute_list_f(df: pd.DataFrame) -> float:
    for c in ["list_f_measure", "f_measure", "F_measure", "f1", "F1", "list_f1"]:
        if c in df.columns:
            return float(pd.to_numeric(df[c], errors="coerce").mean())

    gold_candidates = ["gold_exact", "gold_exact_answer", "exact_answer", "gold"]
    pred_candidates = ["pred_exact", "pred_exact_answer", "pred"]

    gold_col = next((c for c in gold_candidates if c in df.columns), None)
    pred_col = next((c for c in pred_candidates if c in df.columns), None)
    if gold_col is None or pred_col is None:
        return np.nan

    f1s = []
    for _, row in df.iterrows():
        gold = set(norm_text(x) for x in to_list(row[gold_col]) if str(x).strip() != "")
        pred = set(norm_text(x) for x in to_list(row[pred_col]) if str(x).strip() != "")

        if not gold and not pred:
            f1s.append(1.0); continue
        if not gold or not pred:
            f1s.append(0.0); continue

        tp = len(gold & pred)
        prec = tp / len(pred)
        rec  = tp / len(gold)
        f1 = (2 * prec * rec / (prec + rec)) if (prec + rec) else 0.0
        f1s.append(f1)

    return float(np.mean(f1s)) if f1s else np.nan

# ---------- Yes/No Macro F1 ----------
def compute_yesno_macro_f1(df: pd.DataFrame) -> float:
    # 이미 집계된 컬럼이 있으면 우선 사용
    for c in ["yesno_macro_f1", "yesno_f1", "macro_f1", "MacroF1", "exact_yesno_macro_f1"]:
        if c in df.columns:
            return float(pd.to_numeric(df[c], errors="coerce").mean())

    gold_candidates = ["gold_exact", "gold_exact_answer", "exact_answer", "gold"]
    pred_candidates = ["pred_exact", "pred_exact_answer", "pred"]

    gold_col = next((c for c in gold_candidates if c in df.columns), None)
    pred_col = next((c for c in pred_candidates if c in df.columns), None)
    if gold_col is None or pred_col is None:
        return np.nan

    # gold/pred를 라벨(yes/no)로 정규화
    def as_label(v):
        lst = to_list(v)
        if not lst:
            return ""
        s = norm_text(lst[0])
        if "yes" in s:
            return "yes"
        if "no" in s:
            return "no"
        return s  # 혹시 다른 표기가 있으면 그대로

    y_true = [as_label(v) for v in df[gold_col].tolist()]
    y_pred = [as_label(v) for v in df[pred_col].tolist()]

    labels = ["yes", "no"]
    f1s = []
    for lab in labels:
        tp = sum((t == lab and p == lab) for t, p in zip(y_true, y_pred))
        fp = sum((t != lab and p == lab) for t, p in zip(y_true, y_pred))
        fn = sum((t == lab and p != lab) for t, p in zip(y_true, y_pred))
        prec = tp / (tp + fp) if (tp + fp) else 0.0
        rec  = tp / (tp + fn) if (tp + fn) else 0.0
        f1 = (2 * prec * rec / (prec + rec)) if (prec + rec) else 0.0
        f1s.append(f1)

    return float(np.mean(f1s)) if f1s else np.nan

# ---------- summary 생성 ----------
csv_files = sorted(glob.glob(os.path.join(RESULT_DIR, "*.csv")))
rows = []

for fp in csv_files:
    df = read_csv_smart(fp)
    meta = parse_filename(os.path.basename(fp))
    qtype = meta["qtype"]

    rows.append({
        "file": os.path.basename(fp),
        "qtype": qtype,
        "model": meta["model"],
        "pool": meta["pool"],
        "k": meta["k"],
        "factoid_mrr": compute_factoid_mrr(df) if qtype == "factoid" else np.nan,
        "list_f_measure": compute_list_f(df) if qtype == "list" else np.nan,
        "yesno_macro_f1": compute_yesno_macro_f1(df) if qtype == "yesno" else np.nan,
        "n_rows": len(df),
    })

summary = pd.DataFrame(rows)

bad = summary[summary["qtype"].isna()][["file"]]
if len(bad) > 0:
    print("⚠️ 파일명 파싱 실패 목록:")
    print(bad.to_string(index=False))

# ---------- pivots ----------
fact = summary[summary["qtype"] == "factoid"].copy()
fact_pivot = fact.pivot_table(index=["model","pool"], columns="k", values="factoid_mrr", aggfunc="mean").sort_index()

lst = summary[summary["qtype"] == "list"].copy()
list_pivot = lst.pivot_table(index=["model","pool"], columns="k", values="list_f_measure", aggfunc="mean").sort_index()

yn = summary[summary["qtype"] == "yesno"].copy()
yesno_pivot = yn.pivot_table(index=["model","pool"], columns="k", values="yesno_macro_f1", aggfunc="mean").sort_index()

print("=== FACTOID (MRR) PIVOT ===")
display(fact_pivot)

print("\n=== LIST (F-measure) PIVOT ===")
display(list_pivot)

print("\n=== YESNO (Macro F1) PIVOT ===")
display(yesno_pivot)


=== FACTOID (MRR) PIVOT ===


k                    5         10        20
model    pool                              
SBERT    C     0.039755  0.058104  0.076453
         M     0.042813  0.073394  0.088685
         X     0.042813  0.030581  0.036697
bge      C     0.042813  0.082569  0.100917
         CM         NaN       NaN  0.113150
         CX         NaN       NaN  0.082569
         M     0.067278  0.067278  0.113150
         X     0.055046  0.058104  0.042813
         XM         NaN       NaN  0.018349
gemma    C     0.000000  0.000000  0.000000
         M     0.051988  0.045872  0.076453
         X     0.006116  0.009174  0.000000
newbge   C     0.051988  0.061162  0.079511
         M     0.042813  0.055046  0.094801
         X     0.048930  0.039755  0.036697
newsolon C     0.027523  0.058104  0.073394
         M     0.030581  0.061162  0.097859
         X     0.048930  0.042813  0.045872
solon    C     0.039755  0.058104  0.100917
         M     0.036697  0.073394  0.091743
         X     0.045872  0.036697  0.042813


=== LIST (F-measure) PIVOT ===


k                    5         10        20
model    pool                              
SBERT    C     0.000000  0.073350  0.089398
         CM    0.016962  0.105956  0.152274
         CX    0.014740  0.042025  0.142403
         M     0.028316  0.115562  0.161046
         X     0.003451  0.010856  0.095956
         XM    0.000000  0.000000  0.013320
bge      C     0.085065  0.113910  0.174524
         CM    0.079752  0.114422  0.166984
         CX    0.061043  0.072560  0.137666
         M     0.061172  0.089669  0.159260
         X     0.042512  0.056609  0.075718
         XM    0.009899  0.017969  0.023564
gemma    C     0.000000  0.000000  0.010559
         M     0.020291  0.128815  0.164316
         X     0.005887  0.000000  0.026512
llama    C     0.000000  0.001985  0.010695
         CM    0.000000  0.001019  0.036836
         CX    0.000360  0.000000  0.016882
         M     0.000680  0.026585  0.057411
         X     0.000000  0.003418  0.022253
         XM    0.000000  0.000000  0.025615
newbge   C     0.030903  0.122175  0.157721
         M     0.024590  0.116446  0.178677
         X     0.017019  0.017015  0.111743
newsolon C     0.014561  0.101021  0.136190
         M     0.027910  0.162362  0.160328
         X     0.016125  0.020039  0.133885
solon    C     0.025022  0.156314  0.145536
         M     0.037847  0.197585  0.163943
         X     0.016239  0.021318  0.138505


=== YESNO (Macro F1) PIVOT ===


k                    5         10        20
model    pool                              
SBERT    C     0.769800  0.747805  0.753313
         CM    0.726495  0.702598  0.732835
         CX    0.729444  0.705115  0.731147
         M     0.767016  0.662982  0.721842
         X     0.673077  0.652513  0.711807
         XM    0.252776  0.198489  0.256471
bge      C     0.761454  0.742636  0.718615
         CM    0.739317  0.700006  0.763355
         CX    0.723543  0.685844  0.775825
         M     0.730960  0.769534  0.724164
         X     0.689982  0.685105  0.686032
         XM    0.546145  0.590222  0.561030
e5       C     0.743783  0.703504  0.700812
         M     0.743157  0.718699  0.704979
gemma    C     0.187227  0.159390  0.238234
         M     0.687704  0.686202  0.686716
         X     0.422692  0.432191  0.453546
llama    C     0.146348  0.159996  0.215344
         CM    0.491058  0.525338  0.499820
         CX    0.483462  0.450864  0.458995
         M     0.578715  0.584656  0.605611
         X     0.504093  0.498543  0.479558
         XM    0.529139  0.563386  0.497944
mxbai    C     0.750789  0.733406  0.702383
         M     0.758605  0.729524  0.764360
newSBERT C     0.714773  0.706610  0.711106
         M     0.718173  0.702550  0.753299
newbge   C     0.725614  0.640373  0.745035
         M     0.720992  0.657813  0.718219
         X     0.635642  0.690698  0.628780
newsolon C     0.730524  0.744930  0.715720
         M     0.691951  0.733424  0.723282
         X     0.685203  0.738805  0.677396
solon    C     0.710069  0.710843  0.724558
         M     0.738130  0.680768  0.769017
         X     0.663753  0.681042  0.641566